
### Notebook Parameters

- **statement_id**: Comma-separated list of query statement IDs. If set, filters results to these specific queries. Takes priority over time range.
- **query_start_from**: Start UTC timestamp for filtering queries by execution time. Used only if `statement_id` is empty.
- **query_start_to**: End UTC timestamp for filtering queries by execution time. Used only if `statement_id` is empty.
- **warehouse_id**: Warehouse ID to filter queries and cost calculations. If empty, includes all warehouses.
- **executed_by**: Username or service principal to filter queries by who executed them. If empty, includes all users.

In [0]:
-- Cost Per Query Calculator 
-- Excludes warehouse idle time from cost allocation.
-- Only apportions DBUs consumed during active query execution.
-- Uses total_task_duration_ms as the weighted resource consumption metric.
--
-- Usage:
--   Option 1: Set :statement_id for exact lookup (comma-separated for multiple IDs)
--   Option 2: Clear :statement_id, set :query_start_from / :query_start_to for time range
--   Optional: :warehouse_id and :executed_by apply to both options
--
-- NOTE: :statement_id takes priority. If :statement_id is non-empty,
--       :query_start_from and :query_start_to are ignored.
--       Clear :statement_id to use time-range mode.

WITH target_queries AS (
  SELECT
    qh.statement_id,
    qh.executed_by,
    qh.statement_text,
    qh.statement_type,
    qh.execution_status,
    qh.compute.warehouse_id AS warehouse_id,
    qh.start_time,
    qh.end_time,
    qh.total_duration_ms,
    qh.execution_duration_ms,
    qh.total_task_duration_ms,
    qh.read_bytes,
    qh.read_rows,
    qh.produced_rows,
    qh.from_result_cache
  FROM system.query.history qh
  WHERE
    (
      -- Option 1: one or more statement_ids (comma-separated)
      -- Takes priority when non-empty; time-range params are ignored.
      (
        :statement_id != ''
        AND ARRAY_CONTAINS(
          TRANSFORM(SPLIT(:statement_id, ','), s -> TRIM(s)),
          qh.statement_id
        )
      )
      OR (
        -- Option 2: time-range scan (only when :statement_id is empty)
        :statement_id = ''
        AND :query_start_from != ''
        AND qh.start_time >= TRY_CAST(:query_start_from AS TIMESTAMP)
        AND (:query_start_to = '' OR qh.start_time <= TRY_CAST(:query_start_to AS TIMESTAMP))
        AND qh.total_task_duration_ms > 0
        AND qh.execution_status = 'FINISHED'
        AND qh.compute.warehouse_id IS NOT NULL
      )
    )
    -- Optional filters (apply to both options, empty = no filter)
    AND (:executed_by = '' OR qh.executed_by = :executed_by)
    AND (:warehouse_id = '' OR qh.compute.warehouse_id = :warehouse_id)
),

-- For each target query, find overlapping billing windows
target_billing_windows AS (
  SELECT
    tq.statement_id   AS target_stmt_id,
    tq.warehouse_id,
    tq.total_task_duration_ms AS target_task_ms,
    bu.usage_start_time,
    bu.usage_end_time,
    bu.usage_quantity,
    bu.sku_name,
    bu.cloud,
    bu.account_id
  FROM target_queries tq
  JOIN system.billing.usage bu
    ON bu.usage_metadata.warehouse_id = tq.warehouse_id
    AND bu.usage_start_time < tq.end_time
    AND bu.usage_end_time > tq.start_time
  WHERE bu.billing_origin_product = 'SQL'
    AND bu.usage_type = 'COMPUTE_TIME'
),

-- Unique billing windows to process (deduplicated across targets on same warehouse)
unique_windows AS (
  SELECT DISTINCT
    warehouse_id,
    usage_start_time,
    usage_end_time,
    usage_quantity,
    sku_name,
    cloud,
    account_id
  FROM target_billing_windows
),

-- All concurrent queries per unique billing window
-- Explicit WHERE predicates push scan-level filters to system.query.history
-- (redundant with join conditions but helps the optimizer limit scan range)
query_intervals AS (
  SELECT
    uw.warehouse_id,
    uw.usage_start_time,
    uw.usage_end_time,
    uw.usage_quantity,
    uw.sku_name,
    uw.cloud,
    uw.account_id,
    qh.statement_id,
    qh.total_task_duration_ms,
    GREATEST(qh.start_time, uw.usage_start_time) AS q_start,
    LEAST(qh.end_time, uw.usage_end_time)         AS q_end
  FROM unique_windows uw
  JOIN system.query.history qh
    ON qh.compute.warehouse_id = uw.warehouse_id
    AND qh.start_time < uw.usage_end_time
    AND qh.end_time > uw.usage_start_time
    AND qh.total_task_duration_ms > 0
  WHERE
    -- Push warehouse_id down to scan level
    (:warehouse_id = '' OR qh.compute.warehouse_id = :warehouse_id)
    -- Push time bounds to scan level (limits to billing window range)
    AND qh.start_time < (SELECT MAX(usage_end_time) FROM unique_windows)
    AND qh.end_time > (SELECT MIN(usage_start_time) FROM unique_windows)
),

-- =================================================================
-- Interval merging (gaps-and-islands) per warehouse per billing window
-- Computes active time, excluding idle warehouse time
-- =================================================================

distinct_intervals AS (
  SELECT DISTINCT
    warehouse_id,
    usage_start_time,
    usage_end_time,
    q_start,
    q_end
  FROM query_intervals
),

island_detect AS (
  SELECT
    warehouse_id,
    usage_start_time,
    usage_end_time,
    q_start,
    q_end,
    MAX(q_end) OVER (
      PARTITION BY warehouse_id, usage_start_time, usage_end_time
      ORDER BY q_start, q_end
      ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prev_max_end
  FROM distinct_intervals
),

island_groups AS (
  SELECT
    warehouse_id,
    usage_start_time,
    usage_end_time,
    q_start,
    q_end,
    SUM(
      CASE WHEN prev_max_end IS NULL OR q_start > prev_max_end THEN 1 ELSE 0 END
    ) OVER (
      PARTITION BY warehouse_id, usage_start_time, usage_end_time
      ORDER BY q_start, q_end
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS grp
  FROM island_detect
),

merged_intervals AS (
  SELECT
    warehouse_id,
    usage_start_time,
    usage_end_time,
    MIN(q_start) AS active_start,
    MAX(q_end)   AS active_end
  FROM island_groups
  GROUP BY warehouse_id, usage_start_time, usage_end_time, grp
),

window_active_time AS (
  SELECT
    warehouse_id,
    usage_start_time,
    usage_end_time,
    SUM(UNIX_TIMESTAMP(active_end) - UNIX_TIMESTAMP(active_start)) AS active_seconds,
    UNIX_TIMESTAMP(usage_end_time) - UNIX_TIMESTAMP(usage_start_time) AS window_seconds
  FROM merged_intervals
  GROUP BY warehouse_id, usage_start_time, usage_end_time
),

-- =================================================================
-- Proportional allocation (Snowflake-style: idle time excluded)
-- active_dbus = usage_quantity * (active_seconds / window_seconds)
-- query_dbus  = active_dbus * (query_task_ms / total_task_ms_all)
-- =================================================================

concurrent_totals AS (
  SELECT
    qi.warehouse_id,
    qi.usage_start_time,
    qi.usage_end_time,
    qi.usage_quantity,
    qi.sku_name,
    qi.cloud,
    qi.account_id,
    COUNT(DISTINCT qi.statement_id)  AS concurrent_query_count,
    SUM(qi.total_task_duration_ms)   AS total_task_ms_all,
    wat.active_seconds,
    wat.window_seconds,
    qi.usage_quantity * (wat.active_seconds / wat.window_seconds) AS active_dbus
  FROM query_intervals qi
  JOIN window_active_time wat
    ON qi.warehouse_id = wat.warehouse_id
    AND qi.usage_start_time = wat.usage_start_time
    AND qi.usage_end_time = wat.usage_end_time
  GROUP BY
    qi.warehouse_id, qi.usage_start_time, qi.usage_end_time,
    qi.usage_quantity, qi.sku_name, qi.cloud, qi.account_id,
    wat.active_seconds, wat.window_seconds
),

-- Allocate to each target query
allocated AS (
  SELECT
    tbw.target_stmt_id,
    ct.usage_start_time,
    ct.usage_end_time,
    ct.sku_name,
    ct.cloud,
    ct.account_id,
    ct.usage_quantity                                                AS window_total_dbus,
    ct.active_dbus,
    ROUND(ct.usage_quantity - ct.active_dbus, 6)                     AS idle_dbus,
    ct.concurrent_query_count,
    ct.total_task_ms_all                                             AS window_total_task_ms,
    tbw.target_task_ms                                               AS query_task_ms,
    ct.active_seconds,
    ct.window_seconds,
    ROUND(ct.active_seconds / ct.window_seconds * 100, 2)            AS window_active_pct,
    ROUND(tbw.target_task_ms / ct.total_task_ms_all * 100, 2)        AS pct_of_active_work,
    ct.active_dbus * (tbw.target_task_ms / ct.total_task_ms_all)     AS allocated_dbus
  FROM target_billing_windows tbw
  JOIN concurrent_totals ct
    ON tbw.warehouse_id = ct.warehouse_id
    AND tbw.usage_start_time = ct.usage_start_time
    AND tbw.usage_end_time = ct.usage_end_time
  WHERE ct.total_task_ms_all > 0
),

-- Latest warehouse configuration
warehouse_info AS (
  SELECT
    w.warehouse_id,
    w.warehouse_name,
    w.warehouse_type,
    w.warehouse_size
  FROM system.compute.warehouses w
  WHERE w.warehouse_id IN (SELECT DISTINCT warehouse_id FROM target_queries)
  QUALIFY ROW_NUMBER() OVER (PARTITION BY w.warehouse_id ORDER BY w.change_time DESC) = 1
),

-- Cluster count at query time (bounded lookback to avoid full events scan)
warehouse_scale AS (
  SELECT
    tq.statement_id AS target_stmt_id,
    we.cluster_count
  FROM target_queries tq
  JOIN system.compute.warehouse_events we
    ON we.warehouse_id = tq.warehouse_id
    AND we.event_time <= tq.start_time
    AND we.event_time >= tq.start_time - INTERVAL 1 DAY
  WHERE we.event_type IN ('RUNNING', 'SCALED_UP', 'SCALED_DOWN')
  QUALIFY ROW_NUMBER() OVER (PARTITION BY tq.statement_id ORDER BY we.event_time DESC) = 1
)

SELECT
  tq.statement_id,
  tq.executed_by,
  tq.statement_type,
  tq.execution_status,
  wi.warehouse_name,
  wi.warehouse_type,
  wi.warehouse_size,
  COALESCE(ws.cluster_count, 1)                  AS cluster_count_at_query_time,
  tq.start_time                                  AS query_start,
  tq.end_time                                    AS query_end,
  ROUND(tq.total_duration_ms / 1000.0, 2)        AS total_duration_sec,
  ROUND(tq.execution_duration_ms / 1000.0, 2)    AS execution_duration_sec,
  tq.total_task_duration_ms,
  tq.from_result_cache,
  FORMAT_NUMBER(tq.read_bytes / 1048576.0, 2)    AS read_mb,
  FORMAT_NUMBER(tq.read_rows, 0)                 AS read_rows,
  FORMAT_NUMBER(tq.produced_rows, 0)             AS produced_rows,
  -- Diagnostic: billing window breakdown
  MAX(a.window_total_dbus)                       AS window_total_dbus,
  ROUND(MAX(a.active_dbus), 6)                   AS active_dbus_in_window,
  ROUND(MAX(a.idle_dbus), 6)                     AS idle_dbus_excluded,
  MAX(a.window_active_pct)                       AS window_active_pct,
  MAX(a.concurrent_query_count)                  AS concurrent_query_count,
  MAX(a.pct_of_active_work)                      AS pct_of_active_work,
  -- Cost results (idle time excluded)
  ROUND(COALESCE(SUM(a.allocated_dbus), 0), 6)                                     AS allocated_dbus,
  ROUND(COALESCE(SUM(a.allocated_dbus * lp.pricing.effective_list.default), 0), 4)  AS estimated_cost_usd,
  MAX(lp.currency_code)                          AS currency,
  LEFT(tq.statement_text, 500)                   AS statement_preview
FROM target_queries tq
LEFT JOIN allocated a
  ON tq.statement_id = a.target_stmt_id
LEFT JOIN system.billing.list_prices lp
  ON a.sku_name = lp.sku_name
  AND a.cloud = lp.cloud
  AND a.account_id = lp.account_id
  AND lp.price_start_time <= a.usage_start_time
  AND (lp.price_end_time IS NULL OR lp.price_end_time > a.usage_start_time)
  AND lp.currency_code = 'USD'
LEFT JOIN warehouse_info wi
  ON tq.warehouse_id = wi.warehouse_id
LEFT JOIN warehouse_scale ws
  ON tq.statement_id = ws.target_stmt_id
GROUP BY ALL
ORDER BY tq.start_time DESC

In [0]:
-- Cost Per Query PrPr (FE) - provided in https://github.com/databrickslabs/sandbox/tree/main/dbsql/cost_per_query%3APrPr
-- Excludes warehouse idle time from cost allocation using event-driven utilization.
-- Apportions DBUs consumed during active query execution weighted by query work time.
--
-- Usage:
--   Option 1: Set :statement_id for exact lookup (comma-separated for multiple IDs)
--   Option 2: Clear :statement_id, set :query_start_from / :query_start_to for time range
--   Option 3: Clear all to return all queries in the boundary window
--   Optional: :warehouse_id and :executed_by apply to all options
--
-- NOTE: :statement_id takes priority. If :statement_id is non-empty,
--       :query_start_from and :query_start_to are ignored.
--       Clear :statement_id to use time-range mode.

WITH 
-- Must make sure the time window the MV is built on has enough data from ALL 3 tables to generate accurate results (both starts and end time ranges)
table_boundaries AS (
SELECT 
(SELECT MAX(event_time) FROM system.compute.warehouse_events) AS max_events_ts,
(SELECT MAX(end_time) FROM system.query.history) AS max_query_end_ts,
(SELECT MAX(usage_end_time) FROM system.billing.usage) AS max_billing_ts,
(SELECT MIN(event_time) FROM system.compute.warehouse_events) AS min_event_ts,
(SELECT MIN(start_time) FROM system.query.history) AS min_query_start_ts,
(SELECT MIN(usage_end_time) FROM system.billing.usage) AS min_billing_ts,
date_trunc('HOUR', LEAST(max_events_ts, max_query_end_ts, max_billing_ts)) AS selected_end_time,
(date_trunc('HOUR', GREATEST(min_event_ts, min_query_start_ts, min_billing_ts)) + INTERVAL 1 HOUR)::timestamp AS selected_start_time
),

----===== Warehouse Level Calculations =====-----
cpq_warehouse_usage AS (
  SELECT
    usage_metadata.warehouse_id AS warehouse_id,
    *
  FROM
    system.billing.usage AS u
  WHERE
    usage_metadata.warehouse_id IS NOT NULL
    AND usage_start_time >= (SELECT MIN(selected_start_time) FROM table_boundaries)
    AND usage_end_time <= (SELECT MAX(selected_end_time) FROM table_boundaries)
    -- Optional: filter by warehouse_id for performance
    AND (:warehouse_id = '' OR usage_metadata.warehouse_id = :warehouse_id)
),

prices AS (
  select coalesce(price_end_time, date_add(current_date, 1)) as coalesced_price_end_time, *
  from system.billing.list_prices
  where currency_code = 'USD'
),

filtered_warehouse_usage AS (
    -- Warehouse usage is aggregated hourly, that will be the base assumption and grain of allocation moving forward. 
    -- Assume no duplicate records
    SELECT 
      u.warehouse_id warehouse_id,
      date_trunc('HOUR',u.usage_start_time) AS usage_start_hour,
      date_trunc('HOUR',u.usage_end_time) AS usage_end_hour,
      u.usage_quantity AS dbus,
      (
        CAST(p.pricing.effective_list.default AS FLOAT) * dbus
      ) AS usage_dollars
    FROM
      cpq_warehouse_usage AS u
        left join prices as p
        on u.sku_name=p.sku_name
        and u.usage_unit=p.usage_unit
        and (u.usage_end_time between p.price_start_time and p.coalesced_price_end_time)
),

table_bound_expld AS 
(
select timestampadd(hour, h, selected_start_time) as selected_hours
  from table_boundaries
  join lateral explode(sequence(0, timestampdiff(hour, selected_start_time, selected_end_time), 1)) as t (h)
),

----===== Query Level Calculations =====-----
cpq_warehouse_query_history AS (
  SELECT
    account_id,
    workspace_id,
    statement_id,
    executed_by,
    statement_text,
    compute.warehouse_id AS warehouse_id,
    execution_status,
    COALESCE(client_application, 'Unknown') AS client_application,
    (COALESCE(CAST(total_task_duration_ms AS FLOAT) / 1000, 0) +
      COALESCE(CAST(result_fetch_duration_ms AS FLOAT) / 1000, 0) +
      COALESCE(CAST(compilation_duration_ms AS FLOAT) / 1000, 0)
    )  AS query_work_task_time,
    start_time,
    end_time,
    timestampadd(MILLISECOND , coalesce(waiting_at_capacity_duration_ms, 0) + coalesce(waiting_for_compute_duration_ms, 0) + coalesce(compilation_duration_ms, 0), start_time) AS query_work_start_time,
    timestampadd(MILLISECOND, coalesce(result_fetch_duration_ms, 0), end_time) AS query_work_end_time,
    -- NEW - Query source
    CASE
      WHEN query_source.job_info.job_id IS NOT NULL THEN 'JOB'
      WHEN query_source.legacy_dashboard_id IS NOT NULL THEN 'LEGACY DASHBOARD'
      WHEN query_source.dashboard_id IS NOT NULL THEN 'AI/BI DASHBOARD'
      WHEN query_source.alert_id IS NOT NULL THEN 'ALERT'
      WHEN query_source.notebook_id IS NOT NULL THEN 'NOTEBOOK'
      WHEN query_source.sql_query_id IS NOT NULL THEN 'SQL QUERY'
      WHEN query_source.genie_space_id IS NOT NULL THEN 'GENIE SPACE'
      WHEN client_application IS NOT NULL THEN client_application
      ELSE 'UNKNOWN'
    END AS query_source_type,
    COALESCE(
      query_source.job_info.job_id,
      query_source.legacy_dashboard_id,
      query_source.dashboard_id,
      query_source.alert_id,
      query_source.notebook_id,
      query_source.sql_query_id,
      query_source.genie_space_id,
      'UNKNOWN'
    ) AS query_source_id
  FROM
    system.query.history AS h
  WHERE
    statement_type IS NOT NULL
    -- If query touches the boundaries at all, we will divy it up
    AND start_time < (SELECT selected_end_time FROM table_boundaries)
    AND end_time > (SELECT selected_start_time FROM table_boundaries)
    AND total_task_duration_ms > 0 --exclude metadata operations
    AND compute.warehouse_id IS NOT NULL -- = 'd13162f928a069c7'
    -- Optional: filter by warehouse_id for performance
    AND (:warehouse_id = '' OR compute.warehouse_id = :warehouse_id)
)
  ,  cte_warehouse as
(
  select warehouse_id, min(query_work_start_time) as min_start_time
    from cpq_warehouse_query_history
group by warehouse_id
)
,
--- Warehouse + Query Level level allocation
window_events AS (
    SELECT
        warehouse_id,
        event_type,
        event_time,
        cluster_count AS cluster_count,
        CASE
            WHEN cluster_count = 0 THEN 'OFF'
            WHEN cluster_count > 0 THEN 'ON'
        END AS warehouse_state
    FROM system.compute.warehouse_events AS we
    -- Only get window events for when we have query history, otherwise, not usable
    WHERE warehouse_id in (SELECT warehouse_id FROM cte_warehouse)
    AND event_time >= (SELECT timestampadd(day, -1, selected_start_time) FROM table_boundaries)
    AND event_time <= (SELECT selected_end_time FROM table_boundaries)
)
  ,  cte_agg_events_prep as
(
select warehouse_id
     , warehouse_state
     , event_time
     , row_number() over W1
     - row_number() over W2 as grp
  from window_events
window W1 as (partition by warehouse_id                  order by event_time asc)
     , W2 as (partition by warehouse_id, warehouse_state order by event_time asc)
)
  ,  cte_agg_events as
(
  select warehouse_id
       , warehouse_state                                    as window_state
       , min(event_time)                                    as event_window_start
       , lead(min(event_time), 1, selected_end_time) over W as event_window_end
    from cte_agg_events_prep
    join table_boundaries
group by warehouse_id
       , warehouse_state
       , grp
       , selected_end_time
  window W as (partition by warehouse_id order by min(event_time) asc)
)
  ,  cte_all_events as
(
select warehouse_id
     , window_state
     , date_trunc('second', event_window_start) as event_window_start
     , date_trunc('second', event_window_end  ) as event_window_end
  from cte_agg_events
 where date_trunc('second', event_window_start) < date_trunc('second', event_window_end)
 --and date_trunc('second', event_window_start) >= timestamp '2024-11-14 09:00:00'
)
  ,  cte_queries_event_cnt as
(
  select warehouse_id
       , case num
           when 1
           then date_trunc('second', query_work_start_time)
           else timestampadd(second, case when date_trunc('second', query_work_start_time) = date_trunc('second', query_work_end_time) then 1 else 0 end, date_trunc('second', query_work_end_time))
         end as query_event_time
       , sum(num) as num_queries
    from cpq_warehouse_query_history
    join lateral explode(array(1, -1)) as t (num)
group by 1, 2
)
  ,  cte_raw_history as
(
select warehouse_id
     , query_event_time                                    as query_start
     , lead(query_event_time, 1, selected_end_time) over W as query_end
     , sum(num_queries) over W as queries_active
  from cte_queries_event_cnt
  join table_boundaries
window W as (partition by warehouse_id order by query_event_time asc)
)
  ,  cte_raw_history_byday as
(
  select /*+ repartition(64, warehouse_id, query_start_dt) */
         warehouse_id
       , case num
           when 0
           then query_start
           else timestampadd(day, num, query_start::date)
         end::date as query_start_dt
       , case num
           when 0
           then query_start
           else timestampadd(day, num, query_start::date)
         end as query_start
       , case num
           when timestampdiff(day, query_start::date, query_end::date)
           then query_end
           else timestampadd(day, num + 1, query_start::date)
         end as query_end
       , queries_active
    from cte_raw_history
    join lateral explode(sequence(0, timestampdiff(day, query_start::date, query_end::date), 1)) as t (num)
)
  ,  cte_all_time_union as
(
select warehouse_id
     , case num when 1 then event_window_start else event_window_end end ts_start
  from cte_all_events
  join lateral explode(array(1, -1)) as t (num)
 union 
select warehouse_id
     , case num when 1 then query_start else query_end end
  from cte_raw_history_byday
  join lateral explode(array(1, -1)) as t (num)
 union
select warehouse_id, selected_hours
  from cte_warehouse
  join table_bound_expld on true
-- where selected_hours >= timestampadd(day, -1, min_start_time)
)
  ,  cte_periods as
(
select /*+ repartition(64, warehouse_id, dt_start) */
       warehouse_id
     , ts_start::date as dt_start
     , ts_start
     , lead(ts_start, 1, selected_end_time) over W as ts_end
  from cte_all_time_union
  join table_boundaries
window W as (partition by warehouse_id order by ts_start asc)
)
  ,  cte_merge_periods as
(
    select /*+ broadcast(r) */
           p.warehouse_id
         , date_trunc('hour', p.ts_start) as ts_hour
         , sum(timestampdiff(second, p.ts_start, p.ts_end)) as duration
         , case
             when e.window_state = 'OFF'
               or e.window_state is null
             then 'OFF'
             when r.queries_active > 0
             then 'UTILIZED'
             else 'ON_IDLE'
           end as utilization_flag
      from cte_periods           as p
 left join cte_all_events        as e  on e.warehouse_id       = p.warehouse_id
                                      and e.event_window_start < p.ts_end
                                      and e.event_window_end   > p.ts_start
 left join cte_raw_history_byday as r  on r.warehouse_id       = p.warehouse_id
                                      and r.query_start_dt     = p.dt_start
                                      and r.query_start        < p.ts_end
                                      and r.query_end          > p.ts_start
                                      and r.queries_active     > 0
                                      and e.window_state      <> 'OFF'
     where p.ts_start < p.ts_end
  group by all
),

utilization_by_warehouse AS (
  select warehouse_id
       , ts_hour as warehouse_hour
       , coalesce(sum(duration) filter(where utilization_flag = 'UTILIZED'), 0) as utilized_seconds
       , coalesce(sum(duration) filter(where utilization_flag = 'ON_IDLE' ), 0) as idle_seconds
       , coalesce(sum(duration) filter(where utilization_flag = 'OFF'     ), 0) as off_seconds
       , coalesce(sum(duration), 0) as total_seconds
       , try_divide(utilized_seconds, utilized_seconds + idle_seconds)::decimal(3,2) as utilization_proportion
    from cte_merge_periods
group by warehouse_id
       , ts_hour
),

cleaned_warehouse_info AS (
  SELECT
  wu.warehouse_id,
  wu.usage_start_hour AS hour_bucket,
  wu.dbus,
  wu.usage_dollars,
  ut.utilized_seconds,
  ut.idle_seconds,
  ut.total_seconds,
  ut.utilization_proportion
  FROM filtered_warehouse_usage wu
  LEFT JOIN utilization_by_warehouse AS ut ON wu.warehouse_id = ut.warehouse_id -- Join on calculation grain - warehouse/hour
    AND wu.usage_start_hour = ut.warehouse_hour
),

hour_intervals AS (
  -- Generate valid hourly buckets for each query
  SELECT
    statement_id,
    warehouse_id,
    query_work_start_time,
    query_work_end_time,
    query_work_task_time,
    explode(
      sequence(
        0,
        floor((UNIX_TIMESTAMP(query_work_end_time) - UNIX_TIMESTAMP(date_trunc('hour', query_work_start_time))) / 3600)
      )
    ) AS hours_interval,
    timestampadd(hour, hours_interval, date_trunc('hour', query_work_start_time)) AS hour_bucket
  FROM
    cpq_warehouse_query_history
),

statement_proportioned_work AS (
    SELECT * , 
        GREATEST(0,
          UNIX_TIMESTAMP(LEAST(query_work_end_time, timestampadd(hour, 1, hour_bucket))) -
          UNIX_TIMESTAMP(GREATEST(query_work_start_time, hour_bucket))
        ) AS overlap_duration,
        CASE WHEN CAST(query_work_end_time AS DOUBLE) - CAST(query_work_start_time AS DOUBLE) = 0
        THEN 0
        ELSE query_work_task_time * (overlap_duration / (CAST(query_work_end_time AS DOUBLE) - CAST(query_work_start_time AS DOUBLE)))
        END AS proportional_query_work
    FROM hour_intervals
),


attributed_query_work_all AS (
    SELECT
      statement_id,
      hour_bucket,
      warehouse_id,
      SUM(proportional_query_work) AS attributed_query_work
    FROM
      statement_proportioned_work
    GROUP BY
      statement_id,
      warehouse_id,
      hour_bucket
),

--- Cost Attribution
warehouse_time as (
  select
    warehouse_id,
    hour_bucket,
    SUM(attributed_query_work) as total_work_done_on_warehouse
  from
    attributed_query_work_all
  group by
    warehouse_id, hour_bucket
),

-- Create statement_id / hour bucket allocated combinations
history AS (
  SELECT
    a.*,
    b.total_work_done_on_warehouse,
    CASE
      WHEN attributed_query_work = 0 THEN NULL
      ELSE attributed_query_work / total_work_done_on_warehouse
    END AS proportion_of_warehouse_time_used_by_query
  FROM attributed_query_work_all a
    inner join warehouse_time b on a.warehouse_id = b.warehouse_id
              AND a.hour_bucket = b.hour_bucket -- Will only run for completed hours from warehouse usage - nice clean boundary
),

history_with_pricing AS (
  SELECT
    h1.*,
    wh.dbus AS total_warehouse_period_dbus,
    wh.usage_dollars AS total_warehouse_period_dollars,
    wh.utilization_proportion AS warehouse_utilization_proportion,
    wh.hour_bucket AS warehouse_hour_bucket,
    MAX(wh.hour_bucket) OVER() AS warehouse_max_hour_bucket
  FROM
    history AS h1
    LEFT JOIN cleaned_warehouse_info AS wh ON h1.warehouse_id = wh.warehouse_id AND h1.hour_bucket = wh.hour_bucket
),

-- This is at the statement_id / hour grain (there will be duplicates for each statement for each hour bucket the query spans)

query_attribution AS (
  SELECT
    a.*,
    warehouse_max_hour_bucket AS most_recent_billing_hour,
    CASE WHEN warehouse_hour_bucket IS NOT NULL THEN 'Has Billing Record' ELSE 'No Billing Record for this hour and warehouse yet available' END AS billing_record_check,
    CASE
      WHEN total_work_done_on_warehouse = 0 THEN NULL
      ELSE attributed_query_work / total_work_done_on_warehouse
    END AS query_task_time_proportion,

    (warehouse_utilization_proportion * total_warehouse_period_dollars) * query_task_time_proportion  AS query_attributed_dollars_estimation,
    (warehouse_utilization_proportion * total_warehouse_period_dbus) * query_task_time_proportion  AS query_attributed_dbus_estimation
  FROM
    history_with_pricing a
)

-- Final Output
select
      qq.statement_id,
      FIRST(qq.query_source_id) AS query_source_id,
      FIRST(qq.query_source_type) AS query_source_type,
      FIRST(qq.client_application) AS client_application,
      FIRST(qq.executed_by) AS executed_by,
      FIRST(qq.warehouse_id) AS warehouse_id,
      FIRST(qq.statement_text) AS statement_text,
      FIRST(qq.workspace_id) AS workspace_id,
      COLLECT_LIST(NAMED_STRUCT('hour_bucket', qa.hour_bucket, 'hour_attributed_cost', query_attributed_dollars_estimation, 'hour_attributed_dbus', query_attributed_dbus_estimation)) AS statement_hour_bucket_costs,
      FIRST(qq.start_time) AS start_time,
      FIRST(qq.end_time) AS end_time,
      FIRST(qq.query_work_start_time) AS query_work_start_time,
      FIRST(qq.query_work_end_time) AS query_work_end_time,
      COALESCE(timestampdiff(MILLISECOND, FIRST(qq.start_time), FIRST(qq.end_time))/1000, 0) AS duration_seconds,
      COALESCE(timestampdiff(MILLISECOND, FIRST(qq.query_work_start_time), FIRST(qq.query_work_end_time))/1000, 0) AS query_work_duration_seconds,
      FIRST(query_work_task_time) AS query_work_task_time_seconds,
      SUM(query_attributed_dollars_estimation) AS query_attributed_dollars_estimation,
      SUM(query_attributed_dbus_estimation) AS query_attributed_dbus_estimation,
      FIRST(CASE
        WHEN query_source_type = 'JOB' THEN CONCAT('/jobs/', query_source_id)
        WHEN query_source_type = 'SQL QUERY' THEN CONCAT('/sql/queries/', query_source_id)
        WHEN query_source_type = 'AI/BI DASHBOARD' THEN CONCAT('/sql/dashboardsv3/', query_source_id)
        WHEN query_source_type = 'LEGACY DASHBOARD' THEN CONCAT('/sql/dashboards/', query_source_id)
        WHEN query_source_type = 'ALERTS' THEN CONCAT('/sql/alerts/', query_source_id)
        WHEN query_source_type = 'GENIE SPACE' THEN CONCAT('/genie/rooms/', query_source_id)
        WHEN query_source_type = 'NOTEBOOK' THEN CONCAT('/editor/notebooks/', query_source_id)
        ELSE ''
      END) as url_helper,
      FIRST(CONCAT('/sql/history?uiQueryProfileVisible=true&queryId=', qq.statement_id)) AS query_profile_url,
       FIRST(most_recent_billing_hour) AS most_recent_billing_hour,
       FIRST(billing_record_check) AS billing_record_check,
       date_trunc('HOUR', FIRST(qq.start_time)) AS query_start_hour
      from query_attribution qa
      LEFT JOIN cpq_warehouse_query_history AS qq ON qa.statement_id = qq.statement_id -- creating dups of the objects but just re-aggregating
            AND qa.warehouse_id = qq.warehouse_id
      WHERE
        -- Parameter-based filtering (same logic as Genie Code cell)
        (
          -- Option 1: specific statement_ids (comma-separated, takes priority)
          (
            :statement_id != ''
            AND ARRAY_CONTAINS(
              TRANSFORM(SPLIT(:statement_id, ','), s -> TRIM(s)),
              qq.statement_id
            )
          )
          OR (
            -- Option 2: time-range scan (only when :statement_id is empty)
            :statement_id = ''
            AND :query_start_from != ''
            AND qq.start_time >= try_cast(:query_start_from AS TIMESTAMP)
            AND (:query_start_to = '' OR qq.start_time <= try_cast(:query_start_to AS TIMESTAMP))
          )
          OR (
            -- Option 3: no statement_id and no time range = return all
            :statement_id = '' AND :query_start_from = ''
          )
        )
        -- Optional filters (apply to all options, empty = no filter)
        AND (:executed_by = '' OR qq.executed_by = :executed_by)
        AND (:warehouse_id = '' OR qq.warehouse_id = :warehouse_id)
      GROUP BY qq.statement_id

## Cost Per Query — Method Comparison Notes

### Overview

Both cells calculate cost-per-query by excluding warehouse idle time and proportionally allocating only active DBUs. They share the same goal but differ in **how active time is measured**, **what counts as query work**, and **how idle time is detected**.

* **Cost is dependent on concurrent queries** — The same query will be attributed different cost depending on warehouse concurrency at execution time. Low concurrency → higher per-query cost. High concurrency → lower per-query cost. Snowflake's QUERY_ATTRIBUTION_HISTORY exhibits the exact same concurrency dependency. This is not a limitation of Cell 2 or Cell 3 — it's a fundamental, mathematically unavoidable property of any proportional allocation model on shared compute. 
* **Cost can fluctuate by analysis run time** — Due to `system.billing.usage` ingestion lag and (for Cell 3) shifting `table_boundaries` as new data arrives. Cell 2 is more stable once billing data is complete.

---

### Cell 2: Cost Per Query (Genie Code) — Interval-Merging Approach

**Formula:**
```
active_dbus   = usage_quantity × (active_seconds / window_seconds)
allocated_dbus = active_dbus × (query_task_ms / total_task_ms_all)
```

* **Idle detection:** Merges overlapping query `start_time`/`end_time` intervals (gaps-and-islands) to compute `active_seconds`. Denominator is the **full billing window** (e.g. 3600s).
* **Work metric:** `total_task_duration_ms` only.
* **Query timing:** Raw `start_time` / `end_time`.
* **Architecture:** Query-focused — finds target queries → overlapping billing windows → all concurrent queries → proportional allocation.
* **Closest analogue:** Snowflake `QUERY_ATTRIBUTION_HISTORY` (same idle exclusion philosophy, same single weight metric, same raw execution windows).

| Pros | Cons |
| --- | --- |
| Simpler, easier to audit | Blind to warehouse ON/OFF state (no events) |
| Conservative (lower bound) cost estimate | Ignores compilation and result fetch overhead |
| Parameterized for ad-hoc single-query lookups | Uses raw start/end (includes queue time in active window) |
| Stable results once billing data arrives | |

---

### Cell 3: Cost Per Query PrPr (FE) — Event-Driven Utilization Approach

**Formula:**
```
utilization_proportion = utilized_seconds / (utilized_seconds + idle_seconds)
query_attributed_dbus  = utilization_proportion × total_dbus × query_task_time_proportion
```

* **Idle detection:** Uses `system.compute.warehouse_events` to build ON/OFF timeline, cross-references with query activity. Classifies each second as UTILIZED / ON\_IDLE / OFF. Denominator is **ON-time only** (OFF seconds excluded).
* **Work metric:** `total_task_duration_ms + result_fetch_duration_ms + compilation_duration_ms`.
* **Query timing:** Adjusted `query_work_start_time` (excludes wait) / `query_work_end_time` (includes fetch).
* **Architecture:** Warehouse-focused — processes all queries in the boundary window → builds full utilization timeline → attributes cost per hour bucket.

| Pros | Cons |
| --- | --- |
| More precise utilization via warehouse events | More complex, harder to debug |
| Broader work metric captures full query footprint | Higher cost estimates (may over-attribute) |
| Accounts for actual query work windows | Heavier to run (second-level granularity) |
| Designed for batch / MV materialization | Results can shift as new data arrives |
| Rich query source classification | |

---

### Why Cell 3 Produces \~1.6–2x Higher Cost Per Query

Three compounding factors:

1. **Wider query active windows (biggest driver).** Cell 3 uses adjusted `query_work_start_time`/`query_work_end_time` which adds compilation and result fetch to the execution window. For queries with significant `result_fetch_duration_ms`, the work window can be nearly 2x the raw duration. This applies to all concurrent queries, expanding total "utilized" seconds and increasing `utilization_proportion`.

2. **Broader work metric (+\~13%).** Cell 3 weights by `task_ms + compilation_ms + fetch_ms`. If the target query's compile+fetch overhead is proportionally larger than the average concurrent query, it claims a larger share of the allocation pool.

3. **Tighter utilization denominator.** Cell 3's `utilized / (utilized + idle)` excludes OFF seconds. Cell 2's `active / window_seconds` includes OFF time in the denominator. For serverless warehouses with auto-stop, any OFF period within the billing window shrinks Cell 3's denominator further.

These multiply together: wider windows (\~1.5–1.7x) × broader metric (\~1.13x) × tighter denominator (\~1.1x) ≈ **\~2x**.

---

### Snowflake QUERY\_ATTRIBUTION\_HISTORY Alignment

| Dimension | Snowflake | Cell 2 (Genie Code) | Cell 3 (PrPr FE) |
| --- | --- | --- | --- |
| Idle exclusion | Unattributed credit bucket | `idle_dbus_excluded` diagnostic column | Implicit via `1 - utilization_proportion` |
| Idle detection | Query interval merging | Query interval merging (gaps-and-islands) | Warehouse events (ON/OFF state) |
| Weight metric | Execution time only | `total_task_duration_ms` only | `task_ms + compilation_ms + fetch_ms` |
| Query timing | Raw execution window | Raw `start_time` / `end_time` | Adjusted work start/end |
| Architecture | Query-focused, per-window | Query-focused, per-billing-window | Warehouse-focused, full-timeline |

**Cell 2 (Genie Code) is significantly closer to Snowflake's model** — matching on idle exclusion philosophy, single weight metric, raw execution windows, and query-focused architecture.

---

### Warehouse Scaling Behavior

#### How Snowflake Handles Scaling

Snowflake credits are consumed at a rate of `credit_rate × cluster_count × time`. When a warehouse scales from 1 to 3 clusters, the credit burn rate triples instantly. `QUERY_ATTRIBUTION_HISTORY` attributes credits at fine-grained time intervals where the burn rate is known, so queries running during a scaled-up period are automatically attributed from a proportionally larger credit pool. **Scaling is inherent in the billing unit** — no special logic needed.

#### How Cell 2 and Cell 3 Handle Scaling

**Cell 2 does NOT miss the scaling cost.** The total cost of running multiple clusters is fully captured in `system.billing.usage` — if the warehouse ran 2 clusters, `usage_quantity` reflects \~2x the DBUs. Cell 2 uses this as its allocation pool, so all scaling cost enters the formula. What Cell 2 lacks is **intra-window precision**: it cannot distinguish which queries ran during the 1-cluster vs 2-cluster portion of a billing window.

**Cell 3** reads `system.compute.warehouse_events` (which contains `cluster_count`), but reduces it to binary ON/OFF classification (`cluster_count > 0` → ON). It does not use the actual cluster count value for weighting.

| Dimension | Snowflake | Cell 2 (Genie Code) | Cell 3 (PrPr FE) |
| --- | --- | --- | --- |
| Cluster count tracking | Implicit via credit burn rate (per-cluster-second) | Point-in-time lookup only (`warehouse_scale` CTE — diagnostic, not in formula) | Binary ON/OFF only (ignores actual cluster count) |
| Sub-interval breakdown by scale | Yes — fine-grained intervals reflect actual cluster count | No — uses aggregate hourly `usage_quantity` | No — uses aggregate hourly `usage_quantity` |
| Scaling reflected in billing | Direct (credits ∝ clusters × seconds) | Indirect but complete (aggregate DBUs include all clusters) | Indirect but complete (aggregate DBUs include all clusters) |
| Work metric compensation | Not needed (credits already scale) | Partial (`total_task_duration_ms` reflects parallelism across more nodes) | Partial (`query_work_task_time` reflects parallelism) |
| Accuracy during scale events | High (per-interval) | Total cost correct; per-query distribution approximate | Total cost correct; per-query distribution approximate |

#### Example: Scaling Mid-Hour

A warehouse runs 1 cluster for 50 min (Query A), then scales to 3 clusters for 10 min (Query B). Billing shows 10 DBUs total. In reality, \~5 DBUs came from the 1-cluster period and \~5 DBUs from the 3-cluster period (3× rate for 1/5 the time).

* **Snowflake** correctly attributes from the 5-DBU pool for Query A and the 5-DBU pool for Query B.
* **Both Cell 2 and Cell 3** see a single 10-DBU billing record and distribute by task\_ms proportion, blind to the fact that Query B consumed a disproportionate share of DBUs per second during its window.

#### Which Cell Is Closer to Being Fixable?

**Cell 3 is one step away** from handling scaling correctly. It already reads `system.compute.warehouse_events` and tracks every state change with `cluster_count`. The fix would be:

1. Use actual `cluster_count` (not just ON/OFF) in `cte_merge_periods`
2. Weight each sub-interval's DBU allocation by `cluster_count × duration`
3. Distribute the weighted DBUs to queries running in each sub-interval

**Cell 2 would need a more fundamental change** — it has no cluster-count timeline and would need to build one from warehouse events, partially replicating Cell 3's event-driven approach.

#### Why the Impact Is Small in Practice

* **Serverless warehouses** scale quickly, so mixed-scale billing windows are brief
* `total_task_duration_ms` partially self-corrects — a query on more clusters generates more parallel tasks, naturally inflating its task\_ms share
* The total DBU cost is always correct; only the per-query split during scale transitions is approximate

---

### Filter Behavior

* **`:executed_by` and time range filters** — Applied only to the **final output** in both cells. They do NOT affect the cost calculation. All concurrent queries on the warehouse are always considered when computing proportional cost.
* **`:warehouse_id` filter** — Applied **early** in both cells to reduce scan size. Safe because utilization is computed per-warehouse.


## Treatment of Compilation Time & Result Fetching Time

### Architectural Difference: Where These Operations Run

| Operation | Snowflake | Databricks |
| --- | --- | --- |
| **Compilation / Optimization** | Runs on the **Cloud Services layer** (separate from warehouse) | Runs on the **SQL Warehouse compute** (consumes DBUs) |
| **Result Fetching** | Runs on the **Cloud Services layer** (result cache → client transfer) | Runs on the **SQL Warehouse compute** (consumes DBUs) |
| **Execution** | Runs on the **Virtual Warehouse** (consumes credits) | Runs on the **SQL Warehouse compute** (consumes DBUs) |

Snowflake offloads compilation and result fetching to its Cloud Services layer, which is billed separately from warehouse compute credits. Databricks runs all three phases on the warehouse itself, meaning all three consume DBUs within the same billing stream.

---

### How Each Method Treats These Operations

#### Compilation Time

| Aspect | Snowflake | Cell 2 (Genie Code) | Cell 3 (PrPr FE) |
| --- | --- | --- | --- |
| Included in cost weight metric? | **No** — Cloud Services, not warehouse | **No** — uses `total_task_duration_ms` only | **Yes** — adds `compilation_duration_ms` to weight |
| Included in active time window? | **No** — not on warehouse | **Partially** — raw `start_time` includes the compilation period, so it contributes to "active seconds" in interval merging | **No** — `query_work_start_time` is set AFTER compilation (`start_time + wait + compile`) |
| Billed to the user? | Only if Cloud Services exceeds 10% of daily compute credits | **Yes** — warehouse DBUs | **Yes** — warehouse DBUs |

#### Result Fetching Time

| Aspect | Snowflake | Cell 2 (Genie Code) | Cell 3 (PrPr FE) |
| --- | --- | --- | --- |
| Included in cost weight metric? | **No** — Cloud Services, not warehouse | **No** — uses `total_task_duration_ms` only | **Yes** — adds `result_fetch_duration_ms` to weight |
| Included in active time window? | **No** — not on warehouse | **No** — `end_time` is defined as "excluding result fetch time" | **Yes** — `query_work_end_time = end_time + result_fetch_duration_ms` |
| Billed to the user? | Only if Cloud Services exceeds 10% of daily compute credits | **Yes** — warehouse DBUs | **Yes** — warehouse DBUs |

---

### Snowflake Cloud Services Billing — The 10% Adjustment

Snowflake charges for Cloud Services credits (which include compilation and result fetch) **only when they exceed 10% of your daily warehouse compute credits**. Below that threshold, Cloud Services are effectively free. For most workloads, this means compilation and result fetching cost \$0 on Snowflake.

This creates a structural advantage for Snowflake in naive cost comparisons: operations that Databricks bills on the warehouse (and thus appear in cost-per-query) are "hidden" in Snowflake's free Cloud Services tier.

---

### Impact on Cross-Platform Cost Comparison

| Comparison Approach | Recommendation |
| --- | --- |
| **Warehouse compute only** (apples-to-apples) | Use **Cell 2** — matches Snowflake's `CREDITS_ATTRIBUTED_COMPUTE` by excluding compilation and fetch. Note: slightly undercounts Databricks' true warehouse cost. |
| **Total cost of ownership** (apples-to-apples) | Use **Cell 2** for Databricks warehouse cost, then add Snowflake's Cloud Services charges (`SNOWFLAKE.ORGANIZATION_USAGE.METERING_DAILY_HISTORY` where `SERVICE_TYPE = 'CLOUD_SERVICES'`) to the Snowflake side. |
| **Databricks-only attribution** (full accuracy) | Use **Cell 3** — captures the complete warehouse footprint including compilation and fetch, which are real DBU consumers on Databricks. Not comparable to Snowflake's warehouse-only metric. |

---

### Summary

Cell 2 gives Databricks a slight **advantage** in cross-platform comparisons by excluding work that Databricks bills on the warehouse but Snowflake bills on Cloud Services. Cell 3 gives a more **complete** picture of Databricks warehouse cost but produces numbers that are \~1.6–2x higher and not directly comparable to Snowflake's `CREDITS_ATTRIBUTED_COMPUTE`. For a fair total-cost comparison, pair Cell 2 with Snowflake's Cloud Services line item.